In [1]:
#!/usr/bin/env python3
import csv
import io
import requests
from slugify import slugify


DAWUM_API_URL = 'https://api.dawum.de/'
SLUGIFY_GERMAN_UMLAUTS = [
    ['ä', 'ae'],
    ['Ä', 'Ae'],
    ['ö', 'oe'],
    ['Ö', 'Oe'],
    ['ü', 'ue'],
    ['Ü', 'Ue'],
]

rsp = requests.get(DAWUM_API_URL)
data = rsp.json()

# Parse Parliaments
parliaments = {}
for key, value in data['Parliaments'].items():
    parliaments[key] = {
        'parliament_id': slugify(value['Shortcut']),
        'parliament_name': value['Name']
    }

# Parse Institutes
institutes = {}
for key, value in data['Institutes'].items():
    institutes[key] = {
        'institute_id': slugify(value['Name']),
        'institute_name': value['Name'],
    }

# Parse Taskers
taskers = {}
for key, value in data['Taskers'].items():
    taskers[key] = {
        'tasker_id': slugify(value['Name'], replacements=SLUGIFY_GERMAN_UMLAUTS),
        'tasker_name': value['Name'],
    }

# Parse Parties
parties = {}
for key, value in data['Parties'].items():
    parties[key] = {
        'party_id': slugify(value['Shortcut'], replacements=SLUGIFY_GERMAN_UMLAUTS),
        'party_name': value['Name'],
    }

# Parse Surveys
surveys = {}
result_fieldnames = set()
for key, value in data['Surveys'].items():
    surveys[key] = {
        'survey_id': key,
        'survey_date': value['Date'],
        'survey_persons': value['Surveyed_Persons'],
        'survey_start': value['Survey_Period']['Date_Start'],
        'survey_end': value['Survey_Period']['Date_End'],
    }
    surveys[key].update(
        parliaments[value['Parliament_ID']]
    )
    surveys[key].update(
        taskers[value['Tasker_ID']]
    )
    surveys[key].update(
        institutes[value['Institute_ID']]
    )
    for party_id, result in value['Results'].items():
        col = 'result_' + parties[party_id]['party_id']
        result_fieldnames.add(col)
        surveys[key][col] = float(result)


f = io.StringIO()
fieldnames = [
    'survey_id', 'survey_date', 'survey_persons', 'survey_start', 'survey_end',
    'parliament_id', 'parliament_name',
    'institute_id', 'institute_name',
    'tasker_id', 'tasker_name',
]
fieldnames += result_fieldnames
writer = csv.DictWriter(f, fieldnames=fieldnames)
writer.writeheader()
for survey in surveys.values():
    writer.writerow(survey)
print(f.getvalue())

survey_id,survey_date,survey_persons,survey_start,survey_end,parliament_id,parliament_name,institute_id,institute_name,tasker_id,tasker_name,result_volt,result_familie,result_die-partei,result_plus-brandenburg,result_bd,result_bsw,result_afd,result_gruene,result_tierschutzpartei,result_spd,result_npd,result_linke,result_bfth,result_cdu,result_bayernpartei,result_werteunion,result_piraten,result_fdp,result_csu,result_oedp,result_cdu-csu,result_freie-waehler,result_ssw,result_bunt-saar,result_bvb-fw,result_sonstige
4028,2025-12-09,2503,2025-12-02,2025-12-08,bundestag,Bundestag,forsa,Forsa,rtl-n-tv,RTL / n-tv,,,,,,3.0,26.0,12.0,,14.0,,11.0,,,,,,3.0,,,24.0,,,,,7.0
4027,2025-12-09,2007,2025-12-05,2025-12-08,bundestag,Bundestag,insa,INSA,bild,BILD,,,,,,4.0,26.5,10.5,,15.0,,11.0,,,,,,4.0,,,24.5,,,,,4.5
4025,2025-12-06,1206,2025-12-01,2025-12-05,bundestag,Bundestag,insa,INSA,bild-am-sonntag,BILD am Sonntag,,,,,,4.0,26.0,10.0,,15.0,,11.0,,,,,,4.0,,,25.0,,,,,5.0
4024,2025-12-04,1306,2025-12-01,2

In [6]:
import pandas as pd

# Your existing StringIO object 'f'
surveys_df = pd.DataFrame(surveys.values())
surveys_df

,survey_id,survey_date,survey_persons,survey_start,survey_end,parliament_id,parliament_name,tasker_id,tasker_name,institute_id,...,result_die-partei,result_tierschutzpartei,result_werteunion,result_piraten,result_familie,result_plus-brandenburg,result_bfth,result_bunt-saar,result_oedp,result_npd
0,4028,2025-12-09,2503,2025-12-02,2025-12-08,bundestag,Bundestag,rtl-n-tv,RTL / n-tv,forsa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4027,2025-12-09,2007,2025-12-05,2025-12-08,bundestag,Bundestag,bild,BILD,insa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4025,2025-12-06,1206,2025-12-01,2025-12-05,bundestag,Bundestag,bild-am-sonntag,BILD am Sonntag,insa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4024,2025-12-04,1306,2025-12-01,2025-12-03,bundestag,Bundestag,ard-tagesthemen,ARD-Tagesthemen,infratest-dimap,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4023,2025-12-04,1000,2025-11-28,2025-11-30,bundestag,Bundestag,ipsos,Ipsos,ipsos,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3651,374,2017-01-25,2504,2017-01-16,2017-01-20,bundestag,Bundestag,stern-rtl,Stern / RTL,forsa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3652,373,2017-01-24,5028,2017-01-12,2017-01-23,bundestag,Bundestag,spiegel-online,Spiegel Online,civey,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3653,372,2017-01-24,1992,2017-01-20,2017-01-23,bundestag,Bundestag,bild,BILD,insa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3654,370,2017-01-22,2805,2017-01-12,2017-01-18,bundestag,Bundestag,bild-am-sonntag,BILD am Sonntag,verian-emnid,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
surveys_df.columns

Index(['survey_id', 'survey_date', 'survey_persons', 'survey_start',
       'survey_end', 'parliament_id', 'parliament_name', 'tasker_id',
       'tasker_name', 'institute_id', 'institute_name', 'result_afd',
       'result_cdu-csu', 'result_spd', 'result_gruene', 'result_linke',
       'result_sonstige', 'result_fdp', 'result_bsw', 'result_cdu',
       'result_csu', 'result_freie-waehler', 'result_bayernpartei',
       'result_bvb-fw', 'result_bd', 'result_volt', 'result_ssw',
       'result_die-partei', 'result_tierschutzpartei', 'result_werteunion',
       'result_piraten', 'result_familie', 'result_plus-brandenburg',
       'result_bfth', 'result_bunt-saar', 'result_oedp', 'result_npd'],
      dtype='object')

In [13]:
surveys_mainpartys=surveys_df[['survey_id', 'survey_date', 'survey_persons', 'survey_start',
       'survey_end', 'parliament_id', 'parliament_name', 'tasker_id',
       'tasker_name', 'institute_id', 'institute_name', 'result_afd',
       'result_cdu-csu', 'result_spd', 'result_gruene', 'result_linke']]

In [18]:
surveys_mainpartys["institute_name"].unique()

array(['Forsa', 'INSA', 'Infratest dimap', 'Ipsos', 'pollytix', 'Civey',
       'Forschungsgruppe Wahlen', 'YouGov', 'GMS', 'Allensbach',
       'Verian (Emnid)', 'Institut Wahlkreisprognose',
       'Trend Research Hamburg', 'IFM Berlin', 'Policy Matters',
       'Universität Hamburg', 'Mentefactum', 'IM Field', 'uniQma',
       'Conoscope', 'dimap', 'GESS Phone & Field'], dtype=object)

In [22]:
surveys_mainpartys[surveys_mainpartys["parliament_name"]=="Landtag von Baden-Württemberg"].iloc[40:60]

,survey_id,survey_date,survey_persons,survey_start,survey_end,parliament_id,parliament_name,tasker_id,tasker_name,institute_id,institute_name,result_afd,result_cdu-csu,result_spd,result_gruene,result_linke
2125,1902,2021-03-10,2069,2021-03-02,2021-03-09,baden-wurttemberg,Landtag von Baden-Württemberg,spiegel-online,Spiegel Online,civey,Civey,12.0,NaN,13.0,32.0,4.0
2126,1901,2021-03-10,1558,2021-03-01,2021-03-07,baden-wurttemberg,Landtag von Baden-Württemberg,bild,BILD,insa,INSA,12.0,NaN,10.0,32.0,3.0
2133,1894,2021-03-05,1032,2021-03-01,2021-03-04,baden-wurttemberg,Landtag von Baden-Württemberg,zdf-politbarometer,ZDF-Politbarometer,forschungsgruppe-wahlen,Forschungsgruppe Wahlen,11.0,NaN,10.0,35.0,3.0
2134,1896,2021-03-04,1054,2021-02-27,2021-03-04,baden-wurttemberg,Landtag von Baden-Württemberg,institut-wahlkreisprognose,Institut Wahlkreisprognose,institut-wahlkreisprognose,Institut Wahlkreisprognose,13.0,NaN,12.5,33.0,3.5
2137,1891,2021-03-04,1185,2021-03-01,2021-03-03,baden-wurttemberg,Landtag von Baden-Württemberg,ard,ARD,infratest-dimap,Infratest dimap,12.0,NaN,10.0,33.0,4.0
2159,1869,2021-02-13,1009,2021-02-08,2021-02-11,baden-wurttemberg,Landtag von Baden-Württemberg,bild,BILD,insa,INSA,11.0,NaN,11.0,31.0,4.0
2164,1864,2021-02-05,1032,2021-02-01,2021-02-04,baden-wurttemberg,Landtag von Baden-Württemberg,zdf-politbarometer,ZDF-Politbarometer,forschungsgruppe-wahlen,Forschungsgruppe Wahlen,11.0,NaN,10.0,34.0,3.0
2167,1861,2021-02-04,1000,2021-01-29,2021-02-02,baden-wurttemberg,Landtag von Baden-Württemberg,swr-stuttgarter-zeitung,SWR / Stuttgarter Zeitung,infratest-dimap,Infratest dimap,10.0,NaN,11.0,34.0,3.0
2188,1840,2021-01-13,1010,2021-01-05,2021-01-11,baden-wurttemberg,Landtag von Baden-Württemberg,bild,BILD,insa,INSA,12.0,NaN,12.0,30.0,4.0
2204,1824,2020-12-17,1001,2020-12-14,2020-12-16,baden-wurttemberg,Landtag von Baden-Württemberg,swr,SWR,infratest-dimap,Infratest dimap,11.0,NaN,10.0,35.0,3.0


In [6]:
data_not_na=data[~data["einheit_mastr_nummer"].isna()]
data_not_na

,goal100_id,einheit_mastr_nummer,behoerden_einheit_id,name_windpark,name_stromerzeugungseinheit,einheit_betriebsstatus,ags_gemeinde,bundesland,landkreis,gemeinde,...,dauer_genehmigung,dauer_realisierung,id_version_download_datenset_behoerde,id_version_download_datenset_mastr,x_25832,y_25832,x_4326,y_4326,ags_bundesland,ags_landkreis
0,36019,SEE966958743041,NaN,NaN,HS-Flensburg_P00012_W+PV_202kW,In Betrieb,1001000.0,Schleswig-Holstein,Flensburg,Flensburg,...,NaN,NaN,NaN,2.0,529350.868073,6.069506e+06,9.456250,54.771910,1.0,01001
4,79220,SEE970653437198,SH-03000924205-0002,Windpark Lübeck Nord,WEA2 Pöppendorf,In Betrieb,1003000.0,Schleswig-Holstein,Lübeck,Lübeck,...,203.0,539.0,59.0,2.0,619814.179633,5.977998e+06,10.825159,53.936484,1.0,01003
5,77800,SEE930060808431,SH-03000924233-0001,Bültwind Nr.60,Bültwind,In Betrieb,1003000.0,Schleswig-Holstein,Lübeck,Lübeck,...,NaN,NaN,59.0,2.0,620844.345391,5.977568e+06,10.840673,53.932386,1.0,01003
6,78441,SEE948056013141,SH-51001900011-0004,Bürgerwindpark Albersdorf,WEA 4 / V 203772,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,192.0,59.0,7.0,516927.977265,5.998977e+06,9.259109,54.138631,1.0,01051
7,79178,SEE969313089747,SH-51001900011-0001,Bürgerwindpark Albersdorf,WEA 1 / V 203769,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,193.0,59.0,7.0,516647.993158,5.999295e+06,9.254841,54.141498,1.0,01051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42663,41655,SEE905015577458,NaN,meridian Windpark Thonhausen,WEA 4,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,355.0,NaN,48.0,735203.382956,5.636053e+06,12.340000,50.828333,16.0,16077
42664,41828,SEE929841659676,NaN,meridian Windpark Thonhausen,WEA 7,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,240.0,NaN,48.0,735567.702820,5.635576e+06,12.344858,50.823906,16.0,16077
42665,29924,SEE924115847834,NaN,meridian Windpark Thonhausen,WEA 6,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,349.0,NaN,48.0,735559.598844,5.635976e+06,12.345000,50.827500,16.0,16077
42666,24143,SEE924538245743,SN-09000000004-0002,Mark Sahnau / Thonhausen,Mark Sahnau WEA2,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,296.0,34.0,2.0,735756.754790,5.636739e+06,12.348285,50.834268,16.0,16077


In [7]:
genehmigungsdauer=data_not_na[data_not_na["dauer_genehmigung"]>0]
genehmigungsdauer

,goal100_id,einheit_mastr_nummer,behoerden_einheit_id,name_windpark,name_stromerzeugungseinheit,einheit_betriebsstatus,ags_gemeinde,bundesland,landkreis,gemeinde,...,dauer_genehmigung,dauer_realisierung,id_version_download_datenset_behoerde,id_version_download_datenset_mastr,x_25832,y_25832,x_4326,y_4326,ags_bundesland,ags_landkreis
4,79220,SEE970653437198,SH-03000924205-0002,Windpark Lübeck Nord,WEA2 Pöppendorf,In Betrieb,1003000.0,Schleswig-Holstein,Lübeck,Lübeck,...,203.0,539.0,59.0,2.0,619814.179633,5.977998e+06,10.825159,53.936484,1.0,01003
6,78441,SEE948056013141,SH-51001900011-0004,Bürgerwindpark Albersdorf,WEA 4 / V 203772,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,192.0,59.0,7.0,516927.977265,5.998977e+06,9.259109,54.138631,1.0,01051
7,79178,SEE969313089747,SH-51001900011-0001,Bürgerwindpark Albersdorf,WEA 1 / V 203769,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,193.0,59.0,7.0,516647.993158,5.999295e+06,9.254841,54.141498,1.0,01051
8,79324,SEE973804784493,SH-51001900011-0005,Bürgerwindpark Albersdorf,WEA 5 / V 203773,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,200.0,59.0,7.0,517034.002839,5.998625e+06,9.260712,54.135464,1.0,01051
9,78680,SEE954883336339,SH-51001900011-0003,Bürgerwindpark Albersdorf,WEA 3 / V 203771,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,192.0,59.0,7.0,516705.970862,5.998672e+06,9.255694,54.135897,1.0,01051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42651,24577,SEE947239808451,TH-FAWIND-GOAL100-SEE947239808451,Kraasa,Kraasa 9,In Betrieb,16077044.0,Thüringen,Altenburger Land,Starkenberg,...,311.0,412.0,23.0,48.0,729612.900483,5.653407e+06,12.271667,50.986389,16.0,16077
42653,24645,SEE965077486013,TH-FAWIND-GOAL100-SEE965077486013,Kraasa,Kraasa 8,In Betrieb,16077044.0,Thüringen,Altenburger Land,Starkenberg,...,311.0,378.0,23.0,48.0,729668.302083,5.653039e+06,12.272222,50.983056,16.0,16077
42654,24507,SEE926469864444,TH-FAWIND-GOAL100-SEE926469864444,Windpark Kraasa,VE-01-206089,In Betrieb,16077044.0,Thüringen,Altenburger Land,Starkenberg,...,279.0,57.0,23.0,48.0,730314.272403,5.653665e+06,12.281806,50.988417,16.0,16077
42661,24597,SEE953472606747,TH-FAWIND-GOAL100-SEE953472606747,WEA 05,WEA 5,In Betrieb,16077044.0,Thüringen,Altenburger Land,Starkenberg,...,608.0,300.0,23.0,48.0,729947.012664,5.653305e+06,12.276355,50.985338,16.0,16077


In [23]:
median_dauer_bl=genehmigungsdauer.groupby("bundesland")["dauer_genehmigung"].median()
median_dauer_bl

bundesland
Baden-Württemberg         456.0
Brandenburg               579.0
Hessen                    481.0
Mecklenburg-Vorpommern    594.0
Nordrhein-Westfalen       434.0
Rheinland-Pfalz           427.0
Saarland                  366.0
Sachsen                   537.0
Schleswig-Holstein        301.0
Thüringen                 375.0
Name: dauer_genehmigung, dtype: float64

In [24]:
max_dauer_bl=genehmigungsdauer.groupby("bundesland")["dauer_genehmigung"].quantile(.90)
max_dauer_bl

bundesland
Baden-Württemberg         1400.0
Brandenburg               1401.0
Hessen                    1316.5
Mecklenburg-Vorpommern     594.0
Nordrhein-Westfalen       1060.0
Rheinland-Pfalz           1251.4
Saarland                  1115.0
Sachsen                    866.0
Schleswig-Holstein         914.0
Thüringen                 1056.0
Name: dauer_genehmigung, dtype: float64

In [25]:
min_dauer_bl=genehmigungsdauer.groupby("bundesland")["dauer_genehmigung"].quantile(.10)
min_dauer_bl

bundesland
Baden-Württemberg         184.0
Brandenburg               254.0
Hessen                    220.0
Mecklenburg-Vorpommern    594.0
Nordrhein-Westfalen       204.2
Rheinland-Pfalz           138.0
Saarland                  144.0
Sachsen                   447.0
Schleswig-Holstein        163.0
Thüringen                 167.4
Name: dauer_genehmigung, dtype: float64

In [30]:
min_max_bl=pd.merge(min_dauer_bl,max_dauer_bl,on="bundesland",suffixes=("_min","_max"))
min_max_med_bl=pd.merge(median_dauer_bl,min_max_bl,on="bundesland")
min_max_med_bl=min_max_med_bl.reset_index(drop=False)
min_max_med_bl.to_csv("min_max_median_bl.csv",index=False)

In [8]:
genehmigungsdauer_count=genehmigungsdauer[["ags_landkreis","dauer_genehmigung"]].groupby("ags_landkreis").count().reset_index()
genehmigungsdauer_count

,ags_landkreis,dauer_genehmigung
0,01003,1
1,01051,648
2,01053,36
3,01054,709
4,01055,173
...,...,...
154,16073,13
155,16074,25
156,16075,2
157,16076,16


In [9]:
kreise_mindest_list=genehmigungsdauer_count[genehmigungsdauer_count["dauer_genehmigung"]>=5]["ags_landkreis"].tolist()
kreise_mindest_list

['01051',
 '01053',
 '01054',
 '01055',
 '01056',
 '01057',
 '01058',
 '01059',
 '01060',
 '01061',
 '01062',
 '05154',
 '05162',
 '05166',
 '05170',
 '05358',
 '05362',
 '05366',
 '05370',
 '05374',
 '05382',
 '05554',
 '05558',
 '05562',
 '05566',
 '05570',
 '05762',
 '05766',
 '05770',
 '05774',
 '05914',
 '05958',
 '05962',
 '05966',
 '05970',
 '05974',
 '06431',
 '06432',
 '06434',
 '06435',
 '06437',
 '06439',
 '06440',
 '06531',
 '06532',
 '06533',
 '06534',
 '06535',
 '06631',
 '06632',
 '06633',
 '06634',
 '06635',
 '06636',
 '07132',
 '07133',
 '07134',
 '07135',
 '07137',
 '07140',
 '07141',
 '07143',
 '07231',
 '07232',
 '07233',
 '07235',
 '07319',
 '07331',
 '07332',
 '07334',
 '07335',
 '07336',
 '07337',
 '07338',
 '07339',
 '07340',
 '08117',
 '08119',
 '08126',
 '08127',
 '08128',
 '08136',
 '08225',
 '08237',
 '08317',
 '08325',
 '08415',
 '08425',
 '08436',
 '08437',
 '10041',
 '10042',
 '10043',
 '10044',
 '10045',
 '10046',
 '12052',
 '12053',
 '12060',
 '12061',


In [10]:
mindest_zahl_genehmigungen=genehmigungsdauer[genehmigungsdauer["ags_landkreis"].isin(kreise_mindest_list)]
median_time_to_genehmigung=mindest_zahl_genehmigungen[["ags_landkreis","landkreis","dauer_genehmigung"]].groupby(["landkreis","ags_landkreis"]).median().reset_index()
median_time_to_genehmigung

,landkreis,ags_landkreis,dauer_genehmigung
0,Alb-Donau-Kreis,08425,251.0
1,Altenburger Land,16077,311.0
2,Altenkirchen (Westerwald),07132,1733.0
3,Alzey-Worms,07331,244.5
4,Bad Dürkheim,07332,329.0
...,...,...,...
125,Wesel,05170,598.0
126,Westerwaldkreis,07143,372.0
127,Wetteraukreis,06440,644.0
128,Worms,07319,210.0


In [11]:
median_time_to_genehmigung=median_time_to_genehmigung.sort_values(by="dauer_genehmigung")
median_time_to_genehmigung.head(50)

,landkreis,ags_landkreis,dauer_genehmigung
18,Eichsfeld,16061,102.0
72,Ostalbkreis,08136,184.0
25,Freudenstadt,08237,186.0
111,Südliche Weinstraße,07337,186.0
128,Worms,07319,210.0
94,Saale-Holzland-Kreis,16074,211.0
61,Neckar-Odenwald-Kreis,08225,216.0
23,Euskirchen,05366,219.0
89,Rhein-Lahn-Kreis,07141,231.0
3,Alzey-Worms,07331,244.5


In [32]:
median_time_to_genehmigung.head(10).to_csv("shortest_time.csv",index=False)

In [12]:
median_time_to_genehmigung.tail(50)

,landkreis,ags_landkreis,dauer_genehmigung
33,Havelland,12063,542.0
60,Märkischer Kreis,05962,543.0
47,Lahn-Dill-Kreis,06532,544.0
79,Prignitz,12070,555.5
28,Gießen,06531,556.0
67,Oberspreewald-Lausitz,12066,560.0
32,Hagen,05914,562.0
91,Rhein-Sieg-Kreis,05382,570.5
65,Oberbergischer Kreis,05374,576.0
11,Cochem-Zell,07135,592.0


In [13]:
median_time_to_genehmigung.to_csv("median_time_to_genehmigung.csv",index=False) 

In [21]:
dauer_realisierung=data[data["dauer_realisierung"]>0]
dauer_realisierung

,goal100_id,einheit_mastr_nummer,behoerden_einheit_id,name_windpark,name_stromerzeugungseinheit,einheit_betriebsstatus,ags_gemeinde,bundesland,landkreis,gemeinde,...,dauer_genehmigung,dauer_realisierung,id_version_download_datenset_behoerde,id_version_download_datenset_mastr,x_25832,y_25832,x_4326,y_4326,ags_bundesland,ags_landkreis
4,79220,SEE970653437198,SH-03000924205-0002,Windpark Lübeck Nord,WEA2 Pöppendorf,In Betrieb,1003000.0,Schleswig-Holstein,Lübeck,Lübeck,...,203.0,539.0,59.0,2.0,619814.179633,5.977998e+06,10.825159,53.936484,1.0,01003
6,78441,SEE948056013141,SH-51001900011-0004,Bürgerwindpark Albersdorf,WEA 4 / V 203772,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,192.0,59.0,7.0,516927.977265,5.998977e+06,9.259109,54.138631,1.0,01051
7,79178,SEE969313089747,SH-51001900011-0001,Bürgerwindpark Albersdorf,WEA 1 / V 203769,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,193.0,59.0,7.0,516647.993158,5.999295e+06,9.254841,54.141498,1.0,01051
8,79324,SEE973804784493,SH-51001900011-0005,Bürgerwindpark Albersdorf,WEA 5 / V 203773,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,200.0,59.0,7.0,517034.002839,5.998625e+06,9.260712,54.135464,1.0,01051
9,78680,SEE954883336339,SH-51001900011-0003,Bürgerwindpark Albersdorf,WEA 3 / V 203771,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,192.0,59.0,7.0,516705.970862,5.998672e+06,9.255694,54.135897,1.0,01051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42663,41655,SEE905015577458,NaN,meridian Windpark Thonhausen,WEA 4,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,355.0,NaN,48.0,735203.382956,5.636053e+06,12.340000,50.828333,16.0,16077
42664,41828,SEE929841659676,NaN,meridian Windpark Thonhausen,WEA 7,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,240.0,NaN,48.0,735567.702820,5.635576e+06,12.344858,50.823906,16.0,16077
42665,29924,SEE924115847834,NaN,meridian Windpark Thonhausen,WEA 6,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,349.0,NaN,48.0,735559.598844,5.635976e+06,12.345000,50.827500,16.0,16077
42666,24143,SEE924538245743,SN-09000000004-0002,Mark Sahnau / Thonhausen,Mark Sahnau WEA2,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,296.0,34.0,2.0,735756.754790,5.636739e+06,12.348285,50.834268,16.0,16077


In [22]:
realisierungsdauer_count=dauer_realisierung[["ags_landkreis","dauer_realisierung"]].groupby("ags_landkreis").count().reset_index()
realisierungsdauer_count

,ags_landkreis,dauer_realisierung
0,01003,1
1,01051,830
2,01053,60
3,01054,793
4,01055,261
...,...,...
292,16073,4
293,16074,62
294,16075,22
295,16076,14


In [23]:
kreise_mindest_real=realisierungsdauer_count[realisierungsdauer_count["dauer_realisierung"]>=5]["ags_landkreis"]
kreise_mindest_real_list=kreise_mindest_real.tolist()

In [24]:
dauer_realisierung_mindest=dauer_realisierung[dauer_realisierung["ags_landkreis"].isin(kreise_mindest_real_list)]
dauer_realisierung_mindest

,goal100_id,einheit_mastr_nummer,behoerden_einheit_id,name_windpark,name_stromerzeugungseinheit,einheit_betriebsstatus,ags_gemeinde,bundesland,landkreis,gemeinde,...,dauer_genehmigung,dauer_realisierung,id_version_download_datenset_behoerde,id_version_download_datenset_mastr,x_25832,y_25832,x_4326,y_4326,ags_bundesland,ags_landkreis
6,78441,SEE948056013141,SH-51001900011-0004,Bürgerwindpark Albersdorf,WEA 4 / V 203772,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,192.0,59.0,7.0,516927.977265,5.998977e+06,9.259109,54.138631,1.0,01051
7,79178,SEE969313089747,SH-51001900011-0001,Bürgerwindpark Albersdorf,WEA 1 / V 203769,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,193.0,59.0,7.0,516647.993158,5.999295e+06,9.254841,54.141498,1.0,01051
8,79324,SEE973804784493,SH-51001900011-0005,Bürgerwindpark Albersdorf,WEA 5 / V 203773,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,200.0,59.0,7.0,517034.002839,5.998625e+06,9.260712,54.135464,1.0,01051
9,78680,SEE954883336339,SH-51001900011-0003,Bürgerwindpark Albersdorf,WEA 3 / V 203771,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,192.0,59.0,7.0,516705.970862,5.998672e+06,9.255694,54.135897,1.0,01051
10,79070,SEE966072006992,SH-51001900011-0002,Bürgerwindpark Albersdorf,WEA 2 / V 203770,In Betrieb,1051001.0,Schleswig-Holstein,Dithmarschen,Albersdorf,...,258.0,199.0,59.0,7.0,516598.971357,5.998975e+06,9.254073,54.138624,1.0,01051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42663,41655,SEE905015577458,NaN,meridian Windpark Thonhausen,WEA 4,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,355.0,NaN,48.0,735203.382956,5.636053e+06,12.340000,50.828333,16.0,16077
42664,41828,SEE929841659676,NaN,meridian Windpark Thonhausen,WEA 7,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,240.0,NaN,48.0,735567.702820,5.635576e+06,12.344858,50.823906,16.0,16077
42665,29924,SEE924115847834,NaN,meridian Windpark Thonhausen,WEA 6,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,349.0,NaN,48.0,735559.598844,5.635976e+06,12.345000,50.827500,16.0,16077
42666,24143,SEE924538245743,SN-09000000004-0002,Mark Sahnau / Thonhausen,Mark Sahnau WEA2,In Betrieb,16077047.0,Thüringen,Altenburger Land,Thonhausen,...,NaN,296.0,34.0,2.0,735756.754790,5.636739e+06,12.348285,50.834268,16.0,16077


In [26]:
dauer_realisierung_mindest[dauer_realisierung_mindest.landkreis.str.contains("Erfurt")].sort_values(by="dauer_realisierung")

,goal100_id,einheit_mastr_nummer,behoerden_einheit_id,name_windpark,name_stromerzeugungseinheit,einheit_betriebsstatus,ags_gemeinde,bundesland,landkreis,gemeinde,...,dauer_genehmigung,dauer_realisierung,id_version_download_datenset_behoerde,id_version_download_datenset_mastr,x_25832,y_25832,x_4326,y_4326,ags_bundesland,ags_landkreis
41607,24600,SEE954018466771,TH-FAWIND-GOAL100-SEE954018466771,Kerspleben 2,531507,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,176.0,227.0,23.0,2.0,646042.100771,5.654831e+06,11.082553,51.026456,16.0,16051
41608,38145,SEE959393004785,NaN,Kerspleben 1,53474,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,274.0,NaN,2.0,646311.133580,5.654547e+06,11.086272,51.023834,16.0,16051
41575,24529,SEE932471928456,TH-FAWIND-GOAL100-SEE932471928456,CEE Windpark Töttleben,CEE Windpark Töttleben WEA 1,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,407.0,445.0,23.0,48.0,647644.980222,5.655664e+06,11.105733,51.033535,16.0,16051
41578,34940,SEE902132723218,NaN,Windpark Möbisburg GmbH ＆ Co. KG,WEA 1 - Nr. 78418,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,857.0,NaN,2.0,639289.628072,5.641111e+06,10.981073,50.904830,16.0,16051
41580,34945,SEE915327261174,NaN,Windpark Möbisburg GmbH ＆ Co. KG,WEA 3 - Nr. 78420,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,875.0,NaN,2.0,639175.609546,5.641661e+06,10.979662,50.909798,16.0,16051
41579,34943,SEE922411163153,NaN,Windpark Möbisburg GmbH ＆ Co. KG,WEA 2 - Nr. 78419,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,896.0,NaN,2.0,639215.811345,5.641374e+06,10.980124,50.907209,16.0,16051
41587,34950,SEE991418465737,NaN,Windpark Möbisburg GmbH ＆ Co. KG,WEA 7 - Nr. 78424,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,905.0,NaN,2.0,640007.258760,5.641903e+06,10.991577,50.911777,16.0,16051
41583,34949,SEE994547937892,NaN,Windpark Möbisburg GmbH ＆ Co. KG,WEA 6 - Nr. 78423,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,906.0,NaN,2.0,640054.008793,5.641620e+06,10.992133,50.909223,16.0,16051
41596,34952,SEE913659595509,NaN,Windpark Möbisburg GmbH ＆ Co. KG,WEA 9 - Nr. 78426,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,918.0,NaN,2.0,640447.034518,5.642031e+06,10.997878,50.912822,16.0,16051
41593,34954,SEE934461145102,NaN,Windpark Möbisburg GmbH ＆ Co. KG,WEA 11 - Nr. 78428,In Betrieb,16051000.0,Thüringen,Erfurt,Erfurt,...,NaN,919.0,NaN,2.0,641151.260188,5.641966e+06,11.007864,50.912066,16.0,16051


In [90]:
median_time_to_realisierung=dauer_realisierung_mindest[["ags_landkreis","landkreis","dauer_realisierung"]].groupby(["landkreis","ags_landkreis"]).median().reset_index()

median_time_to_genehmigung

,landkreis,ags_landkreis,dauer_genehmigung
0,Alb-Donau-Kreis,08425,251.0
1,Altenburger Land,16077,311.0
2,Altenkirchen (Westerwald),07132,1733.0
3,Alzey-Worms,07331,244.5
4,Bad Dürkheim,07332,329.0
...,...,...,...
125,Wesel,05170,598.0
126,Westerwaldkreis,07143,372.0
127,Wetteraukreis,06440,644.0
128,Worms,07319,210.0


In [91]:
median_time_to_realisierung.sort_values(by="dauer_realisierung").head(50)

,landkreis,ags_landkreis,dauer_realisierung
107,Krefeld,05114,62.0
183,Rhein-Lahn-Kreis,07141,64.0
251,Worms,07319,69.5
7,Ammerland,03451,76.0
46,Dortmund,05913,132.0
61,Freiburg im Breisgau,08311,159.0
216,Sächsische Schweiz-Osterzgebirge,14628,163.0
72,Grafschaft Bentheim,03456,211.0
121,Lüneburg,03355,211.0
3,Altenburger Land,16077,216.0


In [92]:
median_time_to_realisierung.sort_values(by="dauer_realisierung").tail(50)

,landkreis,ags_landkreis,dauer_realisierung
171,Pfaffenhofen a.d. Ilm,09186,483.0
83,Harz,15085,485.0
213,Stendal,15090,486.5
36,Coesfeld,05558,489.0
178,Region Hannover,03241,489.5
117,Lippe,05766,490.0
219,Südwestpfalz,07340,493.0
238,Warendorf,05570,494.5
153,Nürnberger Land,09574,497.0
52,Emden,03402,503.0


In [100]:
genehmigung_realisierung_merged=pd.merge(median_time_to_genehmigung,median_time_to_realisierung,on=["ags_landkreis","landkreis"])
genehmigung_realisierung_merged

,landkreis,ags_landkreis,dauer_genehmigung,dauer_realisierung
0,Alb-Donau-Kreis,08425,251.0,442.0
1,Altenburger Land,16077,311.0,216.0
2,Alzey-Worms,07331,244.5,217.0
3,Bad Dürkheim,07332,329.0,333.0
4,Bad Kreuznach,07133,309.0,472.0
...,...,...,...,...
117,Wesel,05170,598.0,317.0
118,Westerwaldkreis,07143,372.0,350.0
119,Wetteraukreis,06440,644.0,387.0
120,Worms,07319,210.0,69.5


In [108]:
genehmigung_realisierung_merged_rank_1=genehmigung_realisierung_merged.sort_values(by="dauer_genehmigung").reset_index(drop=True).reset_index()
genehmigung_realisierung_merged_rank_1=genehmigung_realisierung_merged_rank_1.rename(columns={"index":"rank_genehmigung"})

In [109]:
genehmigung_realisierung_merged_rank_2=genehmigung_realisierung_merged_rank_1.sort_values(by="dauer_realisierung").reset_index(drop=True).reset_index()
genehmigung_realisierung_merged_rank_2=genehmigung_realisierung_merged_rank_2.rename(columns={"index":"rank_realisierung"})
genehmigung_realisierung_merged_rank_2

,rank_realisierung,rank_genehmigung,landkreis,ags_landkreis,dauer_genehmigung,dauer_realisierung
0,0,8,Rhein-Lahn-Kreis,07141,231.0,64.0
1,1,4,Worms,07319,210.0,69.5
2,2,27,Altenburger Land,16077,311.0,216.0
3,3,29,Oberhavel,12065,316.0,216.5
4,4,9,Alzey-Worms,07331,244.5,217.0
...,...,...,...,...,...,...
117,117,84,Hagen,05914,562.0,826.0
118,118,119,Sigmaringen,08437,1400.0,889.5
119,119,121,Erfurt,16051,2996.0,920.0
120,120,79,Märkischer Kreis,05962,543.0,950.0


In [110]:
genehmigung_realisierung_merged_rank_2["mean_rank"]=genehmigung_realisierung_merged_rank_2[["rank_genehmigung","rank_realisierung"]].mean(axis=1)
genehmigung_realisierung_merged_rank_2

,rank_realisierung,rank_genehmigung,landkreis,ags_landkreis,dauer_genehmigung,dauer_realisierung,mean_rank
0,0,8,Rhein-Lahn-Kreis,07141,231.0,64.0,4.0
1,1,4,Worms,07319,210.0,69.5,2.5
2,2,27,Altenburger Land,16077,311.0,216.0,14.5
3,3,29,Oberhavel,12065,316.0,216.5,16.0
4,4,9,Alzey-Worms,07331,244.5,217.0,6.5
...,...,...,...,...,...,...,...
117,117,84,Hagen,05914,562.0,826.0,100.5
118,118,119,Sigmaringen,08437,1400.0,889.5,118.5
119,119,121,Erfurt,16051,2996.0,920.0,120.0
120,120,79,Märkischer Kreis,05962,543.0,950.0,99.5


In [112]:
genehmigung_realisierung_merged_rank_2.sort_values(by="mean_rank").head(20)

,rank_realisierung,rank_genehmigung,landkreis,ags_landkreis,dauer_genehmigung,dauer_realisierung,mean_rank
1,1,4,Worms,07319,210.0,69.5,2.5
0,0,8,Rhein-Lahn-Kreis,07141,231.0,64.0,4.0
4,4,9,Alzey-Worms,07331,244.5,217.0,6.5
9,9,5,Saale-Holzland-Kreis,16074,211.0,244.0,7.0
20,20,1,Ostalbkreis,08136,184.0,297.0,10.5
17,17,12,Neunkirchen,10043,254.0,283.0,14.5
22,22,7,Euskirchen,05366,219.0,307.0,14.5
2,2,27,Altenburger Land,16077,311.0,216.0,14.5
3,3,29,Oberhavel,12065,316.0,216.5,16.0
12,12,21,Nordfriesland,01054,284.0,266.0,16.5


In [113]:
genehmigung_realisierung_merged_rank_2.sort_values(by="mean_rank").tail(20)

,rank_realisierung,rank_genehmigung,landkreis,ags_landkreis,dauer_genehmigung,dauer_realisierung,mean_rank
92,92,87,Greiz,16076,594.0,453.0,89.5
79,79,101,Potsdam-Mittelmark,12069,694.0,419.0,90.0
78,78,102,Teltow-Fläming,12072,707.0,416.0,90.0
64,64,118,Erzgebirgskreis,14521,1177.0,379.0,91.0
63,63,120,Rhein-Kreis Neuss,05162,1572.5,377.5,91.5
88,88,95,Trier-Saarburg,07235,627.0,444.0,91.5
96,96,89,Olpe,05966,599.0,467.0,92.5
94,94,96,Segeberg,01060,638.0,464.0,95.0
120,120,79,Märkischer Kreis,05962,543.0,950.0,99.5
117,117,84,Hagen,05914,562.0,826.0,100.5


In [114]:
genehmigung_realisierung_merged_rank_2.to_csv("genehmigungen_realisierung.csv",index=False)

In [ ]:
# Viersen called NEW Mönchengladbach
# Erfurt called VSB
#